# Phase 07 — User Simulator

> **📁 Résultat de cette phase :** Les profils d'utilisateurs simulés sont sauvegardés dans `data/processed/user_profiles_1024.parquet` (ou équivalent).

Ce notebook simule le nombre d'utilisateurs actifs par cellule et par créneau temporel.

## 👥 Pourquoi simuler n_users ?
Le dataset Milan original ne contient que les volumes de trafic (Internet, SMS, Appels). Pour le modèle de handover MILP, nous avons besoin de connaître le nombre d'utilisateurs car c'est cette entité qui bascule physiquement entre antennes. 
**Note importante** : Ces données sont simulées pour tester la pipeline. En production, l'opérateur les remplacera par ses données réelles (compteurs RRC_connected par cellule).

## 📈 Modélisation des patterns
La simulation suit trois composantes :
1. **Pattern Journalier** : Deux pics (midi et soir) modélisés par des gaussiennes (comportement humain standard).
2. **Pattern Hebdomadaire** : Boost de trafic durant le week-end.
3. **Bruit** : Bruit gaussien (10%) pour simuler la variabilité réelle.

In [ ]:
import numpy as np
import polars as pl
from pathlib import Path

def simulate_user_series(n_cells=600, n_slots=2976, base_users=50, peak_factor=3.0, seed=42):
    rng   = np.random.default_rng(seed)
    slots = np.arange(n_slots)
    hours = (slots % 48) / 2
    days  = slots // 48
    
    daily_pattern = 1 + peak_factor * (
        np.exp(-((hours - 20)**2) / 8) + 0.3 * np.exp(-((hours - 12)**2) / 4)
    )
    weekend_boost = 1 + 0.3 * ((days % 7) >= 5)
    
    try:
        work_600 = pl.read_parquet('../data/processed/work_600cells.parquet')['square_id'].unique().to_list()
        n_cells = len(work_600)
    except:
        work_600 = list(range(n_cells))
        
    cell_base = base_users * (0.5 + rng.random(n_cells))
    all_data = []
    for i, square_id in enumerate(work_600):
        users = cell_base[i] * daily_pattern * weekend_boost
        users = users + rng.normal(0, users * 0.1)
        users = np.maximum(users, 1).astype(int)
        
        cell_df = pl.DataFrame({
            'square_id': [int(square_id)] * n_slots,
            'slot_30m': slots.astype(int),
            'n_users': users
        })
        all_data.append(cell_df)
    
    return pl.concat(all_data)

print("Simulation en cours...")
df_users = simulate_user_series()

output_path = Path('../data/simulated/n_users_600cells.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_users.write_parquet(output_path)
print(f"Simulation terminée : {len(df_users)} lignes générées.")